# Notebook 1: Read and Join the Tables

This notebook reads every table from the database on its own, checks row counts, keys, and duplicates, then aggregates order_items and order_payments to one row per order before joining everything into a single ML table (one row per order).

**Artifact produced:** `ml_table.csv`

In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [2]:
DB_USER = "olist"
DB_PASSWORD = "olist123"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "olist_db"

In [3]:
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [4]:
# Read every table on its own
orders = pd.read_sql("SELECT * FROM orders", engine)
customers = pd.read_sql("SELECT * FROM customers", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
order_payments = pd.read_sql("SELECT * FROM order_payments", engine)
products = pd.read_sql("SELECT * FROM products", engine)
sellers = pd.read_sql("SELECT * FROM sellers", engine)
category_translation = pd.read_sql("SELECT * FROM category_translation", engine)

In [5]:
print("orders:", orders.shape)
print("customers:", customers.shape)
print("order_items:", order_items.shape)
print("order_payments:", order_payments.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)

orders: (99441, 8)
customers: (99441, 5)
order_items: (112650, 7)
order_payments: (103886, 5)
products: (32951, 9)
sellers: (3095, 4)


In [6]:
# Check row counts, keys, duplicates
print("\nUnique order_id in orders:", orders["order_id"].nunique())
print("Duplicate order_id in orders:", orders["order_id"].duplicated().sum())


Unique order_id in orders: 99441
Duplicate order_id in orders: 0


In [7]:
print("\nRows per order in order_items:")
print(order_items.groupby("order_id").size().describe())


Rows per order in order_items:
count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64


In [8]:
print("\nRows per order in order_payments:")
print(order_payments.groupby("order_id").size().describe())


Rows per order in order_payments:
count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64


In [9]:
# Aggregate order_items to one row per order
items_agg = order_items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
).reset_index()

In [10]:
main_product = (
    order_items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .sort_values("price", ascending=False)
    .drop_duplicates(subset="order_id", keep="first")[["order_id", "product_category_name"]]
)
items_agg = items_agg.merge(main_product, on="order_id", how="left")
items_agg = items_agg.merge(category_translation, on="product_category_name", how="left")

In [11]:
print("\nitems_agg shape:", items_agg.shape)


items_agg shape: (98666, 6)


In [12]:
# Aggregate order_payments to one row per order
payments_agg = order_payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    n_payment_installments=("payment_installments", "max"),
).reset_index()

In [13]:
main_payment_type = (
    order_payments.sort_values("payment_value", ascending=False)
    .drop_duplicates(subset="order_id", keep="first")[["order_id", "payment_type"]]
)
payments_agg = payments_agg.merge(main_payment_type, on="order_id", how="left")

In [14]:
print("payments_agg shape:", payments_agg.shape)

payments_agg shape: (99440, 4)


In [15]:
# Join everything into one ML table, one row per order
ml_table = orders.merge(customers, on="customer_id", how="left")
ml_table = ml_table.merge(items_agg, on="order_id", how="left")
ml_table = ml_table.merge(payments_agg, on="order_id", how="left")

In [16]:
print("\nFinal ml_table shape:", ml_table.shape)
print("Unique orders in ml_table:", ml_table["order_id"].nunique())
print("\nColumns:", list(ml_table.columns))
print("\nSample rows:")
print(ml_table.head(3))


Final ml_table shape: (99441, 20)
Unique orders in ml_table: 99441

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'n_items', 'total_price', 'total_freight', 'product_category_name', 'product_category_name_english', 'total_payment_value', 'n_payment_installments', 'payment_type']

Sample rows:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018

In [17]:
# Save artifact
ml_table.to_csv("ml_table.csv", index=False)
print("\nArtifact saved: ml_table.csv")


Artifact saved: ml_table.csv
